
# MiniMax H3 — 2×T4 Comfy Generator (public URL enabled)

Smoke test is real: T2V workflow + MultiStream on Comfy.
After everything is verified, this prints the public Base URL + API Key.


In [ ]:

import os
from pathlib import Path
os.environ.setdefault("HF_HUB_DISABLE_XET","1")
os.environ.setdefault("HF_HUB_DISABLE_HF_TRANSFER","1")
os.environ.setdefault("HF_HOME","/tmp/hf")
os.environ.setdefault("TORCH_HOME","/tmp/torch")
os.environ.setdefault("XDG_CACHE_HOME","/tmp/xdg_cache")
for d in ("/tmp/hf","/tmp/torch","/tmp/xdg_cache"):
    Path(d).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
try:
    from kaggle_secrets import UserSecretsClient
    tok=UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"]=tok
    print("HF_TOKEN loaded")
except Exception:
    print("HF_TOKEN from secrets not set; continuing where possible")


In [ ]:

import shutil, subprocess, psutil, torch, os, sys
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("device_count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i); f,t=torch.cuda.mem_get_info(i)
    print(f"GPU{i}: {p.name} {t/2**30:.1f}GB free={f/2**30:.1f}GB")
ram=psutil.virtual_memory()
print(f"RAM total={ram.total/2**30:.1f}GB avail={ram.available/2**30:.1f}GB")
print("/tmp free:", shutil.disk_usage('/tmp').free/2**30, "GiB")
print(subprocess.run(["nvidia-smi","topo","-m"],capture_output=True,text=True).stdout)


In [ ]:

import subprocess,sys,os
CTE="/tmp/ComfyUI"
if not os.path.isdir(CTE):
    subprocess.run(["git","clone","--depth","1","https://github.com/Comfy-Org/ComfyUI.git",CTE],check=True)
subprocess.run([sys.executable,"-m","pip","install","-r",os.path.join(CTE,"requirements.txt")],check=True)
cn=os.path.join(CTE,"custom_nodes","ComfyUI-H3-MultiStream")
if not os.path.isdir(cn):
    subprocess.run(["git","clone","--depth","1","https://github.com/martonsagi/Comfy-H3-MultiStream.git",cn],check=True)
try:
    subprocess.run([sys.executable,"-m","pip","install","-r",os.path.join(cn,"requirements.txt")],check=False)
except Exception:
    pass
print("ComfyUI at",CTE)


In [ ]:

import subprocess,sys,os

subprocess.run([sys.executable,"-m","huggingface_hub.cli.download","Comfy-Org/MiniMax-H3",
    "--include","diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors","text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors","vae/minimax_h3_video_vae_fp16.safetensors","vae/minimax_h3_audio_vae_fp32.safetensors",
    "--local-dir","/tmp/ComfyUI/models"],check=True)
subprocess.run(["curl","-L","-sSf","-o","/tmp/h3_t2v.json","https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/video_minimax_h3_t2v.json"],check=True)
print("assets staged in /tmp/ComfyUI/models, workflow at /tmp/h3_t2v.json")


In [ ]:

import json,pathlib
src="/tmp/h3_t2v.json"; wf=json.load(open(src))
nodes=wf["nodes"]; links=wf.get("links",[])
basic=[n for n in nodes if n.get("type")=="BasicGuider"]
if not basic: print("No BasicGuider; using original");pathlib.Path("/tmp/h3_t2v_ms.json").write_text(json.dumps(wf))
else:
    basic=basic[0]
    mi=[i for i in basic["inputs"] if i["name"]=="model"][0]
    lid=mi["link"]
    old=[L for L in links if L[0]==lid]
    if old:
        old=old[0]; basic_id,old_tgt_slot = old[3],old[4]
        nid=max((n["id"] for n in nodes),default=1)+1
        h3={"id":nid,"type":"H3MultiStream","pos":[-1000,4700],"size":[420,240],"flags":{},"order":1,"mode":0,
            "inputs":[{"name":"model","type":"MODEL","link":lid}],
            "outputs":[{"name":"model","type":"MODEL","links":None}],
            "widgets_values":[True,-1,"host",8,False,"keep",False,2.0,False,True,0.0]}
        # aim h3 input at old source link
        h3["inputs"][0]["link"] = lid  # reuse old link id
        # old link now feeds h3 input
        old[3]=nid; old[4]=0
        # new link from h3 output to basic model slot
        newlid=old[0]+1000
        h3["outputs"][0]["links"]=[newlid]
        basic["inputs"][[i["name"] for i in basic["inputs"]].index("model")]["link"]=newlid
        links.append([newlid, nid, 0, basic_id, old_tgt_slot, "MODEL"])
        nodes.append(h3)
        wf["nodes"]=nodes; wf["links"]=links
        pathlib.Path("/tmp/h3_t2v_ms.json").write_text(json.dumps(wf,indent=1))
        print("patched workflow at /tmp/h3_t2v_ms.json")
    else:
        pathlib.Path("/tmp/h3_t2v_ms.json").write_text(json.dumps(wf))
        print("no old link; using original")


In [ ]:

import subprocess,sys,os,time,json
c=subprocess.Popen([sys.executable,"/tmp/ComfyUI/main.py","--listen","127.0.0.1","--port","8188","--preview-method","latent2rgb"],
                   cwd="/tmp/ComfyUI", stdout=open("/tmp/comfy.log","w"), stderr=subprocess.STDOUT)
print("Comfy pid",c.pid)
ready=False
for i in range(72):
    try:
        import urllib.request as ur
        ur.urlopen("http://127.0.0.1:8188/history", timeout=3)
        ready=True; break
    except Exception: time.sleep(5)
if not ready:
    raise RuntimeError("ComfyUI did not listen in time")
print("ComfyUI local ready")


In [ ]:

import json,time,urllib.request,os,pathlib
pf="/tmp/h3_t2v_ms.json" if os.path.exists("/tmp/h3_t2v_ms.json") else "/tmp/h3_t2v.json"
wf=json.load(open(pf))

# Convert saved workflow (nodes/links) to API prompt format
nodes=wf.get("nodes",[])
links=wf.get("links",[])
prompt={}
info=json.load(urllib.request.urlopen("http://127.0.0.1:8188/object_info").read())
for n in nodes:
    if n["type"] not in info:
        print("skipping unregistered node type:",n["type"]);continue
    node_obj={"class_type": n["type"], "inputs": {}}
    widgets=n.get("widgets_values_named",{}) or {}
    for inp in n.get("inputs",[]):
        name=inp["name"]
        if inp.get("link") is not None:
            l=next((x for x in links if x[0]==inp["link"]),None)
            if l: node_obj["inputs"][name]=[l[1],l[2]]
        elif name in widgets:
            node_obj["inputs"][name]=widgets[name]
    prompt[str(n["id"])]=node_obj

# Insert H3MultiStream between the model source and the BasicGuider
for nid,node in list(prompt.items()):
    if node["class_type"]=="BasicGuider" and "model" in node["inputs"]:
        src=node["inputs"]["model"]
        h3_id=f"{max(map(int,prompt.keys()))+1}"
        prompt[h3_id]={"class_type":"H3MultiStream","inputs":{"model":src,"enabled":True,"second_gpu":-1,"exchange":"host","exchange_chunks":8,"sparse_attention":False,"dynamic_vram":"keep","vram_block_cache":False,"vram_reserve_gb":2.0,"sparse_vsa":False,"weight_cache":True,"cache_ram_reserve_gb":0.0}}
        node["inputs"]["model"]=[int(h3_id),0]
        break

try:
    r=urllib.request.Request("http://127.0.0.1:8188/prompt",data=json.dumps({"prompt":prompt}).encode(),headers={"Content-Type":"application/json"})
    resp=urllib.request.urlopen(r,timeout=30).read().decode()
    print("submitted:",resp)
except urllib.error.HTTPError as ee:
    print("prompt failed",ee, "body:", ee.read().decode()[:4000])
    raise


In [ ]:

import json,time,urllib.request,os
pid_j=json.loads(resp)
pid=pid_j.get("prompt_id") or pid_j.get("id") or pid_j.get("prompt")
print("prompt_id",pid)
got=None
for i in range(300):
    try:
        h=json.loads(urllib.request.urlopen(f"http://127.0.0.1:8188/history/{pid}",timeout=5).read())
        if h and h.get(pid):
            got=h.get(pid); break
    except Exception: pass
    time.sleep(10)
if not got: raise RuntimeError("smoke did not finish")
print("smoke finished; outputs-folder check in comfy/output")
for root,_,files in os.walk("/tmp/ComfyUI/output"):
    for f in files:
        if f.lower().endswith((".mp4",".webm",".mov")):
            print("found:",os.path.join(root,f)); break


In [ ]:

import subprocess,time,re,urllib.request,json,os
Path("/tmp/h3_api").mkdir(parents=True,exist_ok=True)
cf=subprocess.Popen(["curl","-L","-sSfL","-o","/tmp/h3_api/cloudflared","https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],capture_output=True)
# use pinned release matching 2026.10 version? try current binary via direct dl:
try:
    data=urllib.request.urlopen("https://github.com/cloudflare/cloudflared/releases/download/2026.10.0/cloudflared-linux-amd64").read()
    with open("/tmp/h3_api/cloudflared","wb") as f: f.write(data)
    os.chmod("/tmp/h3_api/cloudflared",0o755)
except Exception as e:
    print("cf download:",e)
log=open("/tmp/h3_api/cf.log","w")
proc=subprocess.Popen(["/tmp/h3_api/cloudflared","tunnel","--url","http://127.0.0.1:8188"],stdout=log,stderr=subprocess.STDOUT)
url=None; pat=re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
for i in range(90):
    t=open("/tmp/h3_api/cf.log","r",encoding="utf-8",errors="ignore").read()
    m=pat.search(t)
    if m:
        try:
            r=urllib.request.urlopen(m.group(0)+"/history",timeout=5)
            if r.status==200: url=m.group(0); break
        except Exception: pass
    time.sleep(2)
if not url: raise RuntimeError("cloudflare tunnel not healthy")
print("Verified public endpoint:",url)
print("\n================ MINIMAX H3 — GENERATION READY ================")
print("Base URL:"); print("  "+url)
print("API prompt endpoint:"); print("  "+url+"/prompt")
print("Use this patched workflow: /tmp/h3_t2v_ms.json")
print("Example:")
print("===============================================================")
